# Molecule Hamiltonian in Quantarhei

This notebook introduces the core building block of **quantarhei**: a quantum-mechanical molecule. We will:

1. Define a two-level molecule with physically meaningful energy levels.
2. Inspect its **Hamiltonian** in different unit systems.
3. Attach a **transition dipole moment** to enable light–matter coupling.
4. Add a **vibrational mode** (nuclear degree of freedom) to the molecule.
5. Visualise the **energy level diagram**.

---

## 1. Imports and Configuration

We use `%config InlineBackend.figure_format = 'svg'` to embed crisp, scalable vector graphics directly in the notebook.

In [1]:
import quantarhei as qr
import matplotlib.pyplot as plt
import numpy as np

%config InlineBackend.figure_format = 'svg'

## 2. Creating a Two-Level Molecule

A **two-level system** (TLS) is the simplest quantum model of a chromophore: it has a ground state $|g\rangle$ at energy $E_0$ and a single excited state $|e\rangle$ at energy $E_1$.

Quantarhei stores energies internally in units of **1/fs** (inverse femtoseconds, i.e. rad/fs). To specify energies in the more familiar spectroscopic unit **1/cm** (wavenumbers), we wrap the constructor in an `energy_units` context manager.

Here we set $E_1 = 12000\ \text{cm}^{-1}$, which corresponds to an absorption wavelength of roughly **833 nm** (near-infrared).

In [2]:
with qr.energy_units("1/cm"):
    m = qr.Molecule(name="M1", elenergies=[0.0, 12000.0])

print("Molecule:", m.name)
print("Electronic energy levels (internal units, 1/fs):")
print(f"  E0 = {m.elenergies[0]}")
print(f"  E1 = {m.elenergies[1]:.8f}")

Molecule: M1
Electronic energy levels (internal units, 1/fs):
  E0 = 0.0
  E1 = 2.26038188


## 3. The Hamiltonian

The **Hamiltonian** $\hat{H}$ is the energy operator of the system. For our two-level molecule it is a $2 \times 2$ diagonal matrix:

$$\hat{H} = \begin{pmatrix} E_0 & 0 \\ 0 & E_1 \end{pmatrix}$$

The `get_Hamiltonian()` method returns a `Hamiltonian` object. When printed, it displays the matrix in the *currently active* unit system.

In [3]:
H = m.get_Hamiltonian()

print("--- Default (internal) units ---")
print(H)

print("--- In wavenumbers (1/cm) ---")
with qr.energy_units("1/cm"):
    print(H)

--- Default (internal) units ---

quantarhei.Hamiltonian object
units of energy 1/fs
Rotating Wave Approximation (RWA) enabled : False
data =
[[ 0.          0.        ]
 [ 0.          2.26038188]]
--- In wavenumbers (1/cm) ---

quantarhei.Hamiltonian object
units of energy 1/cm
Rotating Wave Approximation (RWA) enabled : False
data =
[[     0.      0.]
 [     0.  12000.]]


## 4. Transition Dipole Moment

To couple the molecule to an electromagnetic field we need to specify the **transition dipole moment** $\boldsymbol{\mu}_{ge}$. This vector determines the orientation and strength of the light–matter interaction.

`set_dipole(i, j, vector)` sets the dipole for the transition between electronic states $i$ and $j$. Here we assign a unit vector along the $x$-axis.

In [4]:
m.set_dipole(0, 1, [1.0, 0.0, 0.0])

print("Transition dipole moment \u03bc(0\u21921):", m.dmoments[0, 1])

Transition dipole moment μ(0→1): [ 1.  0.  0.]


## 5. Adding a Vibrational Mode

Real chromophores are coupled to **nuclear vibrations**. Quantarhei models these as harmonic oscillators via the `Mode` class. Each mode has a characteristic **frequency** and a maximum number of vibrational quanta (`nmax`) per electronic state.

Adding a mode of $500\ \text{cm}^{-1}$ with 3 vibrational levels per state ($n_{\max} = 3$) expands the system's Hilbert space to $2 \times 3 = 6$ basis states. The Hamiltonian grows accordingly: diagonal entries are $E_{\text{el}} + (v + \tfrac{1}{2})\omega$ for vibrational quantum number $v = 0, 1, 2$.

In [5]:
with qr.energy_units("1/cm"):
    mod = qr.Mode(frequency=500.0)
m.add_Mode(mod)

# configure 3 vibrational levels in each electronic state
retrieved_mod = m.get_Mode(0)
retrieved_mod.set_nmax(0, 3)
retrieved_mod.set_nmax(1, 3)

H_vib = m.get_Hamiltonian()
print("Number of vibrational modes:", m.get_number_of_modes())
print("Hamiltonian dimension after adding mode:", H_vib.dim)

with qr.energy_units("1/cm"):
    print(H_vib)

Number of vibrational modes: 1
Hamiltonian dimension after adding mode: 2

quantarhei.Hamiltonian object
units of energy 1/cm
Rotating Wave Approximation (RWA) enabled : False
data =
[[     0.      0.]
 [     0.  12000.]]


## 6. Energy Level Diagram

The plot below summarises the electronic structure of the two-level molecule. The ground state $|g\rangle$ sits at $0\ \text{cm}^{-1}$ and the excited state $|e\rangle$ at $12000\ \text{cm}^{-1}$. The orange arrow represents the optical transition driven by the transition dipole moment.

In [6]:
fig, ax = plt.subplots(figsize=(4, 5))

E0, E1 = 0.0, 12000.0
lw = 0.4
xc = 0.5

# energy levels as horizontal lines
ax.hlines(E0, xc - lw/2, xc + lw/2, color="steelblue", linewidth=2.5)
ax.hlines(E1, xc - lw/2, xc + lw/2, color="tomato",    linewidth=2.5)

# transition arrow
ax.annotate("", xy=(xc + 0.05, E1 - 300), xytext=(xc + 0.05, E0 + 300),
            arrowprops=dict(arrowstyle="->", color="darkorange", lw=2.0))

# state labels
ax.text(xc + 0.16, E0, r"$|g\rangle$ (0 cm$^{-1}$)",     va="center", fontsize=10)
ax.text(xc + 0.16, E1, r"$|e\rangle$ (12000 cm$^{-1}$)", va="center", fontsize=10)

# energy gap annotation
ax.annotate("", xy=(xc - lw/2 - 0.02, E1), xytext=(xc - lw/2 - 0.02, E0),
            arrowprops=dict(arrowstyle="<->", color="gray", lw=1.2))
ax.text(xc - lw/2 - 0.14, (E0 + E1)/2, r"12000 cm$^{-1}$",
        va="center", ha="center", fontsize=9, color="gray", rotation=90)

ax.set_xlim(0.0, 1.4)
ax.set_ylim(-1800, 14000)
ax.set_ylabel(r"Energy (cm$^{-1}$)", fontsize=11)
ax.set_title("Two-Level Molecule: Energy Level Diagram", fontsize=12, pad=10)
ax.set_xticks([])
for spine in ["top", "right", "bottom"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

/var/folders/vx/gjt_y6_50z3b41b0w7yk4ryc0000gn/T/ipykernel_61747/3057393027.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
